# Week 54, Day 2: Overfitting and underfitting

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w054/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The U-shaped test error

For each degree 0 to 12, fit np.polyfit to each of the 100 training sets and record the training MSE and the test MSE on the fixed test set. Use the median over training sets for the test error, because a few high-degree fits explode.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(542)
f = lambda x: 0.8 * np.sin(np.pi * x)
sets = [(xt := r.uniform(-1, 1, 15), f(xt) + 0.3 * r.normal(size=15)) for _ in range(100)]
x_te = r.uniform(-1, 1, 2000); y_te = f(x_te) + 0.3 * r.normal(size=2000)
tr_err, te_err = np.zeros(13), np.zeros(13)
for deg in range(13):
    tr, te = [], []
    for xt, yt in sets:
        w = np.polyfit(xt, yt, deg)
        tr.append(...)
        te.append(...)
    tr_err[deg], te_err[deg] = np.mean(tr), np.median(te)
best_deg = int(np.argmin(te_err))
for d in [0, 1, 3, 5, 8, 12]:
    print(f"degree {d:2d}: training {tr_err[d]:.3f}, test {te_err[d]:.3f}")
print(f"best degree by test error: {best_deg}")
plt.semilogy(tr_err, "o-", label="training"); plt.semilogy(te_err, "o-", label="test (median)"); plt.axhline(0.09, color="grey", ls="--")
plt.xlabel("degree"); plt.ylabel("MSE"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_shapes():
    "tr_err and te_err have one value per degree"
    assert tr_err.shape == (13,) and te_err.shape == (13,)
def test_train():
    "training error never increases with degree"
    assert np.all(np.diff(tr_err) <= 1e-9)
def test_best():
    "best_deg minimises the median test error and is between 3 and 6"
    assert best_deg == np.argmin(te_err) and 3 <= best_deg <= 6

test_shapes()
test_train()
test_best()
print("All checks passed")

### Exercise: An encoding model that fits noise

Write r2(y, yhat). For each p, fit least squares (with an intercept) to the first p pixels of the 60 training trials, and compute R² on the training trials and on 2,000 new test trials.

In [ ]:
P, n_tr = 60, 60
w_true = np.zeros(P); w_true[:5] = [1.0, 0.8, -0.6, 0.5, -0.4]
X_tr, X_te = r.normal(size=(n_tr, P)), r.normal(size=(2000, P))
y_tr = X_tr @ w_true + 1.0 * r.normal(size=n_tr); y_te = X_te @ w_true + 1.0 * r.normal(size=2000)

def r2(y, yhat):
    return ...

ps = [5, 10, 20, 40, 55]
tr_r2, te_r2 = np.zeros(len(ps)), np.zeros(len(ps))
for k, p in enumerate(ps):
    A_tr, A_te = np.c_[X_tr[:, :p], np.ones(n_tr)], np.c_[X_te[:, :p], np.ones(2000)]
    w, *_ = np.linalg.lstsq(A_tr, y_tr, rcond=None)
    tr_r2[k], te_r2[k] = ...
    print(f"{p:2d} features: training R² {tr_r2[k]:.3f}, test R² {te_r2[k]:.3f}")

In [ ]:
import numpy as np
def test_r2_formula():
    "r2(y, yhat) is one minus the residual sum of squares over the total sum of squares"
    yy = np.array([1.0, 2, 3, 4]); assert abs(r2(yy, yy + 0.5) - (1 - 1.0 / 5.0)) < 1e-12
def test_train():
    "training R² rises with the number of features"
    assert np.all(np.diff(tr_r2) > 0)
def test_test():
    "test R² is best with 5 features and below zero with 55"
    assert np.argmax(te_r2) == 0 and te_r2[-1] < 0

test_r2_formula()
test_train()
test_test()
print("All checks passed")

### Exercise: All the features, kept in check

Write ridge_fit(A, y, lam), the ridge solution with the last column (the intercept) unpenalised. Fit the 55-feature model for each lambda and record its test R².

In [ ]:
def ridge_fit(A, y, lam):
    D = np.eye(A.shape[1]); D[-1, -1] = 0           # do not penalise the intercept
    return ...

lams = np.logspace(-2, 3, 21)
A_tr, A_te = np.c_[X_tr[:, :55], np.ones(n_tr)], np.c_[X_te[:, :55], np.ones(2000)]
ridge_test = np.array([r2(y_te, A_te @ ridge_fit(A_tr, y_tr, lam)) for lam in lams])
print(f"best lambda {lams[np.argmax(ridge_test)]:.1f}: test R² {ridge_test.max():.3f} (unpenalised: {te_r2[-1]:.3f}; 5 true features: {te_r2[0]:.3f})")
plt.semilogx(lams, ridge_test, "o-"); plt.axhline(0, color="grey"); plt.xlabel("lambda"); plt.ylabel("test R²"); plt.show()

In [ ]:
import numpy as np
def test_ridge():
    "ridge_fit returns the penalised least-squares weights (intercept unpenalised)"
    A = np.c_[X_tr[:, :55], np.ones(n_tr)]; D = np.eye(56); D[-1, -1] = 0
    assert np.allclose(ridge_fit(A, y_tr, 3.0), np.linalg.solve(A.T @ A + 3.0 * D, A.T @ y_tr))
def test_rescue():
    "with the best penalty, the 55-feature model has a positive test R² above 0.3"
    assert ridge_test.max() > 0.3 and lams[np.argmax(ridge_test)] > 1

test_ridge()
test_rescue()
print("All checks passed")